In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os
import json
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')


In [8]:
# (CHANGE THESE ACCORDING TO YOUR DRIVE)
STUDENTLIFE_PATH = "/content/drive/MyDrive/AAI/Data"
#OUTPUT_PATH = "/content/drive/MyDrive/AI_Burnout_Predictor/results_realistic_studentlife"

#os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Paths configured.")


Paths configured.


In [9]:
# Loading StudentLife data
def load_studentlife_json(folder_path):
    all_data = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            student_id = file_name.replace(".json", "")
            with open(os.path.join(folder_path, file_name), "r") as f:
                records = json.load(f)
                for r in records:
                    r["student_id"] = student_id
                    all_data.append(r)
    return pd.DataFrame(all_data)

print("Loading StudentLife...")

stress_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Stress"))
activity_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Activity"))
sleep_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Sleep"))

print(f"Stress: {stress_raw.shape}")
print(f"Activity: {activity_raw.shape}")
print(f"Sleep: {sleep_raw.shape}")


Loading StudentLife...
Stress: (2408, 5)
Activity: (833, 9)
Sleep: (1644, 7)


In [10]:
stress_raw.head()

,level,location,resp_time,student_id,null
0,2,"43.70644884,-72.28820168",1364609268,Stress_u10,NaN
1,1,"43.70241112,-72.28788985",1364800742,Stress_u10,NaN
2,1,"43.70243257,-72.28772434",1364770637,Stress_u10,NaN
3,NaN,NaN,1364121099,Stress_u10,4
4,NaN,NaN,1364118564,Stress_u10,"43.70621142,-72.28697323"


In [11]:
activity_raw.head()

,Social2,null,resp_time,student_id,other_relaxing,other_working,relaxing,working,location
0,2,1,1365015406,Activity_u30,NaN,NaN,NaN,NaN,NaN
1,2,2,1364584997,Activity_u30,NaN,NaN,NaN,NaN,NaN
2,2,1,1364757470,Activity_u30,NaN,NaN,NaN,NaN,NaN
3,3,1,1365015405,Activity_u30,NaN,NaN,NaN,NaN,NaN
4,3,1,1365285715,Activity_u30,NaN,NaN,NaN,NaN,NaN


In [12]:
sleep_raw.head()

,null,resp_time,student_id,hour,location,rate,social
0,1,1364121437,Sleep_u54,NaN,NaN,NaN,NaN
1,1,1364118985,Sleep_u54,NaN,NaN,NaN,NaN
2,1,1364121435,Sleep_u54,NaN,NaN,NaN,NaN
3,1,1364121434,Sleep_u54,NaN,NaN,NaN,NaN
4,1,1364118978,Sleep_u54,NaN,NaN,NaN,NaN


In [13]:
# Cleaning the STRESS DATASET
# =========================
print("\n DATASET: STRESS")
print("Shows self-reported student stress levels over time")

print("\nCleaning StudentLife Stress...")

stress_clean = stress_raw.copy()

# Dropping the 'null' column
if 'null' in stress_clean.columns:
    stress_clean = stress_clean.drop(columns=['null'])

print("\nInitial Stress Dataset:")
print(stress_clean)

# Converting  timestamp
stress_clean['timestamp'] = pd.to_datetime(stress_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(stress_clean)

# Cleaning  student_id
stress_clean['student_id'] = stress_clean['student_id'].str.replace('Stress_', '')
print("\nAfter cleaning student_id:")
print(stress_clean)



 DATASET: STRESS
Shows self-reported student stress levels over time

Cleaning StudentLife Stress...

Initial Stress Dataset:
     level                  location   resp_time  student_id
0        2  43.70644884,-72.28820168  1364609268  Stress_u10
1        1  43.70241112,-72.28788985  1364800742  Stress_u10
2        1  43.70243257,-72.28772434  1364770637  Stress_u10
3      NaN                       NaN  1364121099  Stress_u10
4      NaN                       NaN  1364118564  Stress_u10
...    ...                       ...         ...         ...
2403     1  43.69230566,-72.26694933  1369244504  Stress_u44
2404     1  43.69213441,-72.26712004  1369329213  Stress_u44
2405     1  43.69222786,-72.26719913  1369416038  Stress_u44
2406     2   43.69341383,-72.2733575  1369675715  Stress_u44
2407     5  43.69232465,-72.26672118  1370034952  Stress_u44

[2408 rows x 4 columns]

After converting resp_time to timestamp:
     level                  location   resp_time  student_id  \
0        2

In [14]:
#renaming the level column
stress_clean = stress_clean.rename(columns={'level': 'stress_level'})
print("\nAfter renaming level → stress_level:")
print(stress_clean)

#Converting stress_level to numeric data
stress_clean['stress_level'] = pd.to_numeric(stress_clean['stress_level'], errors='coerce')
print("\nAfter converting stress_level to numeric:")
print(stress_clean)

if 'null' in stress_clean.columns:

    # if stress_level is missing but null looks like numeric, then use this
    null_as_num = pd.to_numeric(stress_clean['null'], errors='coerce')
    fill_mask = stress_clean['stress_level'].isna() & null_as_num.notna()
    if fill_mask.any():
        stress_clean.loc[fill_mask, 'stress_level'] = null_as_num.loc[fill_mask]

    #If location is missing but null looks like "lat,long", then use this
    if 'location' in stress_clean.columns:
        null_as_str = stress_clean['null'].astype(str)
        coord_mask = stress_clean['location'].isna() & null_as_str.str.match(
            r'^-?\d+(\.\d+)?,-?\d+(\.\d+)?$'
        )
        if coord_mask.any():
            stress_clean.loc[coord_mask, 'location'] = stress_clean.loc[coord_mask, 'null']

print("\nAfter recovering values from 'null' (if applicable):")
print(stress_clean)

#dropping missing stress values
stress_clean = stress_clean.dropna(subset=['stress_level'])
print("\nAfter dropping NaN stress levels:")
print(stress_clean)

#Explicit float conversion
stress_clean['stress_level'] = stress_clean['stress_level'].astype(float)
print("\nFinal cleaned Stress dataset:")
print(stress_clean)



After renaming level → stress_level:
     stress_level                  location   resp_time student_id  \
0               2  43.70644884,-72.28820168  1364609268        u10   
1               1  43.70241112,-72.28788985  1364800742        u10   
2               1  43.70243257,-72.28772434  1364770637        u10   
3             NaN                       NaN  1364121099        u10   
4             NaN                       NaN  1364118564        u10   
...           ...                       ...         ...        ...   
2403            1  43.69230566,-72.26694933  1369244504        u44   
2404            1  43.69213441,-72.26712004  1369329213        u44   
2405            1  43.69222786,-72.26719913  1369416038        u44   
2406            2   43.69341383,-72.2733575  1369675715        u44   
2407            5  43.69232465,-72.26672118  1370034952        u44   

               timestamp  
0    2013-03-30 02:07:48  
1    2013-04-01 07:19:02  
2    2013-03-31 22:57:17  
3    2013-03-

In [15]:
# Code for Activity data cleaning
print("\n DATASET: ACTIVITY")
print(" Shows students' daily activity levels (social, working, relaxing)")

print("\nCleaning StudentLife Activity")

activity_clean = activity_raw.copy()

# Drop the 'null' coloumns
if 'null' in activity_clean.columns:
    activity_clean = activity_clean.drop(columns=['null'])

print("\nInitial Activity Dataset:")
print(activity_clean)

# Convert timestamp

activity_clean['timestamp'] = pd.to_datetime(activity_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(activity_clean)

# Clean student_id
activity_clean['student_id'] = activity_clean['student_id'].str.replace('Activity_', '')
print("\nAfter cleaning student_id:")
print(activity_clean)



 DATASET: ACTIVITY
 Shows students' daily activity levels (social, working, relaxing)

Cleaning StudentLife Activity

Initial Activity Dataset:
    Social2   resp_time    student_id other_relaxing other_working relaxing  \
0         2  1365015406  Activity_u30            NaN           NaN      NaN   
1         2  1364584997  Activity_u30            NaN           NaN      NaN   
2         2  1364757470  Activity_u30            NaN           NaN      NaN   
3         3  1365015405  Activity_u30            NaN           NaN      NaN   
4         3  1365285715  Activity_u30            NaN           NaN      NaN   
..      ...         ...           ...            ...           ...      ...   
828     NaN  1368744712  Activity_u58              1             2        5   
829     NaN  1368237572  Activity_u58              1             1        5   
830     NaN  1368571581  Activity_u58              1             2        5   
831     NaN  1368828978  Activity_u58              3             

In [16]:


# key columns of studentlife dataset

for col in ["Social2", "working", "other_working", "relaxing", "other_relaxing"]:
    if col not in activity_clean.columns:
        activity_clean[col] = np.nan



# Converting to numeric

for col in ["Social2", "working", "other_working", "relaxing", "other_relaxing"]:
  activity_clean[col] = pd.to_numeric(activity_clean[col], errors="coerce")



# Computing interpretable scores

activity_clean["workload_score"] = activity_clean[["working", "other_working"]].sum(axis=1, min_count=1)
activity_clean["recovery_score"] = activity_clean[["relaxing", "other_relaxing"]].sum(axis=1, min_count=1)
activity_clean["social_score"] = activity_clean["Social2"]


print("\nAfter computing workload_score / recovery_score / social_score:")

print(activity_clean[["student_id", "timestamp", "workload_score", "recovery_score", "social_score"]].head())



# Keep relevant columns and drop rows where ALL scores are missing

activity_clean = activity_clean[["student_id", "timestamp", "workload_score", "recovery_score", "social_score"]]
activity_clean = activity_clean.dropna( subset=["workload_score", "recovery_score", "social_score"], how="all" ).copy()



print("\nFinal cleaned Activity dataset (3 scores):")
print(activity_clean.head())




After computing workload_score / recovery_score / social_score:
  student_id           timestamp  workload_score  recovery_score  social_score
0        u30 2013-04-03 18:56:46             NaN             NaN           2.0
1        u30 2013-03-29 19:23:17             NaN             NaN           2.0
2        u30 2013-03-31 19:17:50             NaN             NaN           2.0
3        u30 2013-04-03 18:56:45             NaN             NaN           3.0
4        u30 2013-04-06 22:01:55             NaN             NaN           3.0

Final cleaned Activity dataset (3 scores):
  student_id           timestamp  workload_score  recovery_score  social_score
0        u30 2013-04-03 18:56:46             NaN             NaN           2.0
1        u30 2013-03-29 19:23:17             NaN             NaN           2.0
2        u30 2013-03-31 19:17:50             NaN             NaN           2.0
3        u30 2013-04-03 18:56:45             NaN             NaN           3.0
4        u30 2013-04-0

In [17]:
# Sleep dataset processing

# Print dataset name
print("\ DATASET: SLEEP")

# Describe dataset purpose
print("Shows students' self-reported sleep duration in hours")

# Start cleaning process
print("\nCleaning StudentLife Sleep...")

# Create a copy of raw dataset
sleep_clean = sleep_raw.copy()

# Remove irrelevant null column if present
if 'null' in sleep_clean.columns:
    sleep_clean = sleep_clean.drop(columns=['null'])

# Display initial dataset
print("\nInitial Sleep Dataset:")
print(sleep_clean)

# Convert response time to timestamp
sleep_clean['timestamp'] = pd.to_datetime(sleep_clean['resp_time'], unit='s')

# Show dataset after timestamp conversion
print("\nAfter converting resp_time to timestamp:")
print(sleep_clean)

\ DATASET: SLEEP
Shows students' self-reported sleep duration in hours

Cleaning StudentLife Sleep...

Initial Sleep Dataset:
       resp_time student_id hour                  location rate social
0     1364121437  Sleep_u54  NaN                       NaN  NaN    NaN
1     1364118985  Sleep_u54  NaN                       NaN  NaN    NaN
2     1364121435  Sleep_u54  NaN                       NaN  NaN    NaN
3     1364121434  Sleep_u54  NaN                       NaN  NaN    NaN
4     1364118978  Sleep_u54  NaN                       NaN  NaN    NaN
...          ...        ...  ...                       ...  ...    ...
1639  1369329215  Sleep_u44   12  43.69213441,-72.26712004    1      3
1640  1369416050  Sleep_u44   11  43.69222786,-72.26719913    1      4
1641  1369503822  Sleep_u44   11  43.69476866,-72.27943547    1      4
1642  1369675713  Sleep_u44   10   43.69341383,-72.2733575    1      4
1643  1370114619  Sleep_u44   12  43.69232465,-72.26672118    1      3

[1644 rows x 6 column

In [18]:
# Clean student_id by removing prefix
sleep_clean['student_id'] = sleep_clean['student_id'].str.replace('Sleep_', '')

# Show dataset after cleaning student_id
print("\nAfter cleaning student_id:")
print(sleep_clean)

# Convert hour column to numeric sleep_hours
sleep_clean['sleep_hours'] = pd.to_numeric(sleep_clean['hour'], errors='coerce')

# Show dataset after converting sleep hours
print("\nAfter converting hour to sleep_hours:")
print(sleep_clean)

# Recover missing sleep hours from null column if available
if 'null' in sleep_clean.columns:
    null_as_num = pd.to_numeric(sleep_clean['null'], errors='coerce')
    fill_mask = sleep_clean['sleep_hours'].isna() & null_as_num.notna()
    if fill_mask.any():
        sleep_clean.loc[fill_mask, 'sleep_hours'] = null_as_num.loc[fill_mask]

# Show dataset after recovery step
print("\nAfter recovering sleep_hours from null if applicable:")
print(sleep_clean)

# Keep only relevant columns and remove missing values
sleep_clean = sleep_clean[['student_id', 'timestamp', 'sleep_hours']].dropna()

# Display final cleaned dataset
print("\nFinal cleaned Sleep dataset:")
print(sleep_clean)


After cleaning student_id:
       resp_time student_id hour                  location rate social  \
0     1364121437        u54  NaN                       NaN  NaN    NaN   
1     1364118985        u54  NaN                       NaN  NaN    NaN   
2     1364121435        u54  NaN                       NaN  NaN    NaN   
3     1364121434        u54  NaN                       NaN  NaN    NaN   
4     1364118978        u54  NaN                       NaN  NaN    NaN   
...          ...        ...  ...                       ...  ...    ...   
1639  1369329215        u44   12  43.69213441,-72.26712004    1      3   
1640  1369416050        u44   11  43.69222786,-72.26719913    1      4   
1641  1369503822        u44   11  43.69476866,-72.27943547    1      4   
1642  1369675713        u44   10   43.69341383,-72.2733575    1      4   
1643  1370114619        u44   12  43.69232465,-72.26672118    1      3   

               timestamp  
0    2013-03-24 10:37:17  
1    2013-03-24 09:56:25  
2 

In [19]:
# Sahitya_week2 added code for feature engineering for stress dataset
print("CREATING WEEKLY STUDENTLIFE FEATURES (STRESS + SLEEP + ACTIVITY)")

for df, time_col in [  # loop through each dataframe and its timestamp column name

    (stress_clean, "timestamp"),  # pair: stress dataframe + timestamp column

    (activity_clean, "timestamp"),  # pair: activity dataframe + timestamp column

    (sleep_clean, "timestamp"),  # pair: sleep dataframe + timestamp column

]:

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")  # convert to datetime;

stress_clean["stress_level"] = pd.to_numeric(stress_clean["stress_level"], errors="coerce")  # convert stress to numeric;

for col in ["workload_score","recovery_score","social_score"]:  # iterate through activity score columns

    if col in activity_clean.columns:  # check if the column exists in activity data

        activity_clean[col] = pd.to_numeric(activity_clean[col], errors="coerce")  # convert that column to numeric;

sleep_clean["sleep_hours"] = pd.to_numeric(sleep_clean["sleep_hours"], errors="coerce")  # convert sleep hours to numeric;

sleep_clean.head()

CREATING WEEKLY STUDENTLIFE FEATURES (STRESS + SLEEP + ACTIVITY)


,student_id,timestamp,sleep_hours
6,u54,2013-03-28 17:01:16,5.0
7,u54,2013-03-30 08:27:32,6.0
8,u54,2013-03-31 18:11:15,4.0
9,u54,2013-04-05 17:18:54,5.0
10,u54,2013-04-10 01:34:20,5.0


the following code is written by jaimil kothari - week 2

In [20]:


def add_student_zscore(df, id_col, value_col, new_col, time_col=None):  # to define helper function to add a z-score column

    if time_col is None:  # checking if a time column name was provided or not
        # falling back to index order if timestamp isn't provided
        time_col = None  # keeping it none explicitly

    if time_col is not None and time_col in df.columns:  # if time_col exists then doing sort by id + time
        df = df.sort_values([id_col, time_col]).copy()  # sorting data to ensure expanding stats follow time order

    else:  # else sort only by student id
        df = df.sort_values([id_col]).copy()  # sorting by student id for deterministic order

    def _exp_z(s: pd.Series) -> pd.Series:  # this is an inner function to compute expanding z-score for one student series
        exp_mean = s.expanding(min_periods=1).mean()  # expanding mean up to each point
        exp_std = s.expanding(min_periods=2).std().fillna(0.0)  # expanding std; needs >=2 points
        exp_std = exp_std.replace(0, np.nan).fillna(1.0)  # replacing 0 std with 1 to avoid divide-by-zero
        return (s - exp_mean) / exp_std  # computing z-score using expanding mean/std


    df[new_col] = df.groupby(id_col)[value_col].transform(_exp_z)  # expanding z-score per student
    return df  # returning dataframe with new z-score column added


stress_clean = add_student_zscore(stress_clean, "student_id", "stress_level", "stress_z", time_col="timestamp")  # adding stress_z

for col, zcol in [("workload_score","workload_z"),("recovery_score","recovery_z"),("social_score","social_z")]:  # mapping activity cols to z cols
    if col in activity_clean.columns:  # true only if the activity column exists
        activity_clean = add_student_zscore(activity_clean, "student_id", col, zcol, time_col="timestamp")  # adding corresponding z-score column

sleep_clean = add_student_zscore(sleep_clean, "student_id", "sleep_hours", "sleep_z")  # adding sleep_z (no time_col passed here)

In [21]:
stress_clean.head()

,stress_level,location,resp_time,student_id,timestamp,stress_z
1819,2.0,"43.70692415,-72.2873929",1364237696,u00,2013-03-25 18:54:56,0.000000
1820,2.0,"43.70555193,-72.28704778",1364268806,u00,2013-03-26 03:33:26,0.000000
1821,2.0,"43.70555193,-72.28704778",1364268814,u00,2013-03-26 03:33:34,0.000000
1822,1.0,"43.70678675,-72.28732051",1364346740,u00,2013-03-27 01:12:20,-1.500000
1824,1.0,"43.70508322,-72.28677496",1364437527,u00,2013-03-28 02:25:27,-1.095445


In [22]:
sleep_clean.head(100)

,student_id,timestamp,sleep_hours,sleep_z
706,u00,2013-04-05 17:00:15,4.0,0.000000
697,u00,2013-03-28 02:25:12,6.0,0.707107
696,u00,2013-03-27 18:18:27,8.0,1.000000
695,u00,2013-03-27 04:09:49,8.0,0.783349
694,u00,2013-03-26 18:07:14,7.0,0.239046
...,...,...,...,...
445,u02,2013-04-08 17:58:10,7.0,-0.707107
443,u02,2013-04-06 20:49:39,7.0,-0.672908
442,u02,2013-04-05 17:09:28,8.0,0.046639
441,u02,2013-04-04 17:33:02,8.0,0.045332




---


The following code is done by Md Israk Hossain, Week 2


---





In [23]:

# ISO weeks are a standard calendar system used worldwide. They keep “week numbers” consistent (weeks starting on Monday), making weekly grouping more reliable.
stress_clean["week"] = stress_clean["timestamp"].dt.isocalendar().week.astype(int)  # taking the ISO week number from each stress timestamp and storing it as an int
activity_clean["week"] = activity_clean["timestamp"].dt.isocalendar().week.astype(int)  # adding ISO week number for each activity record
sleep_clean["week"] = sleep_clean["timestamp"].dt.isocalendar().week.astype(int)  # adding ISO week number for each sleep record


# starting to build a new dataframe containing weekly stress summaries. Here we are computing multiple summary stats for each student-week group
stress_weekly = (
    stress_clean.groupby(["student_id", "week"]).agg(
        stress_mean=("stress_level", "mean"),  # calculating average stress level during that week
        stress_std=("stress_level", "std"),  # measuring how much stress varies during that week
        stress_max=("stress_level", "max"),  # finding the highest stress level recorded in that week
        stress_count=("stress_level", "count"),  # counting how many stress readings exist in that week
        stress_z_mean=("stress_z", "mean"),  # calculating average normalized stress (z-score) in that week
        stress_z_std=("stress_z", "std"),  # measuring variation of normalized stress (z-score) in that week
        stress_z_max=("stress_z", "max"), ).reset_index() ) # finding the maximum normalized stress (z-score) in that week


agg_dict = {}  # creating a dictionary to store aggregation rules

for prefix, val_col, z_col in [  # looping through each activity score and its z-score column
    ("workload", "workload_score", "workload_z"),
    ("recovery", "recovery_score", "recovery_z"),
    ("social", "social_score", "social_z")]:

    if val_col in activity_clean.columns:  # checking if the raw score column exists
        agg_dict.update({  # adding aggregation rules for raw values
            f"{prefix}_mean": (val_col, "mean"),  # calculating weekly average
            f"{prefix}_std": (val_col, "std"),  # measuring weekly variation
            f"{prefix}_max": (val_col, "max"),  # finding weekly maximum value
            f"{prefix}_count": (val_col, "count"), }) # counting number of records in that week


    if z_col in activity_clean.columns:  # checking if normalized z-score column exists. If yes, we add aggregation rules for z-score values
        agg_dict.update({
            f"{prefix}_z_mean": (z_col, "mean"),  # calculating weekly average z-score
            f"{prefix}_z_std": (z_col, "std"),  # measuring weekly variation of z-scores
            f"{prefix}_z_max": (z_col, "max"),})  # finding highest z-score in that week

# building weekly activity summary dataframe
activity_weekly = (
    activity_clean.groupby(["student_id", "week"])  # grouping activity rows by student and week
    .agg(**agg_dict)  # applying all prepared aggregation rules
    .reset_index())   # converting grouped index back into normal columns

# building weekly sleep summary dataframe
sleep_weekly = (
    sleep_clean.groupby(["student_id", "week"]).agg(
        sleep_mean=("sleep_hours", "mean"),  # calculating weekly average sleep hours
        sleep_std=("sleep_hours", "std"),  # measuring variation of sleep hours during the week
        sleep_count=("sleep_hours", "count"),  # counting sleep records in that week
        sleep_z_mean=("sleep_z", "mean"),  # calculating weekly average normalized sleep (z-score)
        sleep_z_std=("sleep_z", "std"), ).reset_index())  # measuring variation of normalized sleep (z-score)

# building final feature table (one row per student per week)
studentlife_features = (
    stress_weekly  # using weekly stress table as the base
    .merge(activity_weekly, on=["student_id", "week"], how="left")  # attaching weekly activity features
    .merge(sleep_weekly, on=["student_id", "week"], how="left")  # attaching weekly sleep features
)

# final table containing the StudentLife multimodal weekly dataset (stress + activity + sleep together)
multimodal_df = studentlife_features.copy()  # creating a copy to avoid accidental modification of the original

In [24]:
multimodal_df.head()

,student_id,week,stress_mean,stress_std,stress_max,stress_count,stress_z_mean,stress_z_std,stress_z_max,workload_mean,...,social_max,social_count,social_z_mean,social_z_std,social_z_max,sleep_mean,sleep_std,sleep_count,sleep_z_mean,sleep_z_std
0,u00,13,2.705882,1.358524,5.0,17,0.477033,1.154053,2.035527,NaN,...,3.0,5.0,0.203169,0.926565,1.434274,7.083333,1.621354,12.0,0.415912,0.677284
1,u00,14,2.181818,1.470930,5.0,11,-0.252880,1.063170,1.784366,NaN,...,2.0,6.0,-0.184553,0.705472,0.333333,6.666667,3.011091,6.0,0.194802,1.128202
2,u00,15,2.166667,0.983192,3.0,6,-0.223616,0.708273,0.420137,5.400000,...,2.0,1.0,0.402200,NaN,0.402200,7.000000,3.391165,9.0,0.327429,1.286125
3,u00,16,3.333333,0.516398,4.0,6,0.645965,0.375674,1.147061,4.333333,...,NaN,0.0,NaN,NaN,NaN,5.000000,2.738613,5.0,-0.443324,1.169954
4,u00,17,1.000000,0.000000,1.0,4,-1.146164,0.040471,-1.100449,5.200000,...,NaN,0.0,NaN,NaN,NaN,5.750000,2.500000,4.0,-0.587543,1.373933


In [25]:
# Sahitya_week3 Created Target Variable

# Define proxy burnout risk label (multi-signal, 2-week-ahead)

print("CREATING TARGET LABEL (PROXY)")

# Two-week-ahead outcomes (ONLY for the label, never used as input features)

multimodal_df["future_stress_2w"] = multimodal_df.groupby("student_id")["stress_mean"].shift(-2)

multimodal_df["future_sleep_2w"]  = multimodal_df.groupby("student_id")["sleep_mean"].shift(-2)



# Keep only rows where we can define the 2-week-ahead outcome

multimodal_df = multimodal_df.dropna(subset=["future_stress_2w", "future_sleep_2w"]).reset_index(drop=True)



# Data-adaptive thresholds (quantile search)

# We try a small grid of quantiles and pick the pair that yields a reasonable positive rate.

# Target: ~10% to ~30% positives (enough signal, but not too many).

stress_q_candidates = [0.65, 0.70, 0.75, 0.80]

sleep_q_candidates  = [0.35, 0.30, 0.25, 0.20]



target_rate = 0.15

min_rate, max_rate = 0.10, 0.30



best = None  # (score, stress_q, sleep_q, stress_th, sleep_th, pos_rate, pos_count)



for sq in stress_q_candidates:

    stress_th = multimodal_df["future_stress_2w"].quantile(sq)

    for lq in sleep_q_candidates:

        sleep_th = multimodal_df["future_sleep_2w"].quantile(lq)



        y = ((multimodal_df["future_stress_2w"] >= stress_th) &

             (multimodal_df["future_sleep_2w"]  <= sleep_th)).astype(int)



        pos_rate = float(y.mean())

        pos_count = int(y.sum())



        # Score: prefer rates in [min_rate, max_rate] and close to target_rate

        in_range = (min_rate <= pos_rate <= max_rate)

        score = abs(pos_rate - target_rate) + (0 if in_range else 0.5)  # penalty if out of range



        if best is None or score < best[0]:

            best = (score, sq, lq, float(stress_th), float(sleep_th), pos_rate, pos_count)



_, STRESS_Q, SLEEP_Q, STRESS_TH, SLEEP_TH, POS_RATE, POS_COUNT = best



multimodal_df["burnout_risk"] = (

    (multimodal_df["future_stress_2w"] >= STRESS_TH) &

    (multimodal_df["future_sleep_2w"]  <= SLEEP_TH)

).astype(int)



print("Label created: burnout_risk")

print(f"Chosen stress quantile: {STRESS_Q}  -> STRESS_TH = {STRESS_TH:.3f}")

print(f"Chosen sleep  quantile: {SLEEP_Q}  -> SLEEP_TH  = {SLEEP_TH:.3f}")

print(f"Positive rate: {POS_RATE:.3f}  |  Positives: {POS_COUNT} / {len(multimodal_df)}")

print("Class distribution:")

print(multimodal_df["burnout_risk"].value_counts(dropna=False))



CREATING TARGET LABEL (PROXY)
Label created: burnout_risk
Chosen stress quantile: 0.65  -> STRESS_TH = 2.512
Chosen sleep  quantile: 0.35  -> SLEEP_TH  = 7.000
Positive rate: 0.164  |  Positives: 34 / 207
Class distribution:
burnout_risk
0    173
1     34
Name: count, dtype: int64




---



In [26]:
# Prepare feature sets (baseline vs multimodal)

print("="*70)

print("PREPARING FEATURE SETS")

print("="*70)



y = multimodal_df["burnout_risk"].astype(int)



# Three feature sets (Ablation Study)

# A) Stress-only

# B) Stress + Sleep

# C) Stress + Sleep + Activity (Full Multimodal)



baseline_cols = [

    # raw stress stats

    "stress_mean","stress_std","stress_max","stress_count",

    # normalized stress stats (deviation from personal baseline)

    "stress_z_mean","stress_z_std","stress_z_max",

    # temporal stress features (raw + z)

    "stress_mean_diff1","stress_mean_diff2",

    "stress_z_mean_diff1","stress_z_mean_diff2",

    "stress_mean_roll4_mean","stress_mean_roll4_std",

    "stress_z_mean_roll4_mean","stress_z_mean_roll4_std",

    # missingness flags

    "stress_missing"

]



stress_sleep_cols = baseline_cols + [

    "sleep_mean","sleep_std","sleep_count",

    "sleep_z_mean","sleep_z_std",

    "sleep_mean_diff1","sleep_mean_diff2",

    "sleep_z_mean_diff1","sleep_z_mean_diff2",

    "sleep_mean_roll4_mean","sleep_mean_roll4_std",

    "sleep_z_mean_roll4_mean","sleep_z_mean_roll4_std",

]



multimodal_cols = stress_sleep_cols + [

    # Workload (working-related)

    "workload_mean","workload_std","workload_max","workload_count",

    "workload_z_mean","workload_z_std","workload_z_max",

    "workload_mean_diff1","workload_mean_diff2",

    "workload_z_mean_diff1","workload_z_mean_diff2",

    "workload_mean_roll4_mean","workload_mean_roll4_std",

    "workload_z_mean_roll4_mean","workload_z_mean_roll4_std",



    # Recovery (relaxing-related)

    "recovery_mean","recovery_std","recovery_max","recovery_count",

    "recovery_z_mean","recovery_z_std","recovery_z_max",

    "recovery_mean_diff1","recovery_mean_diff2",

    "recovery_z_mean_diff1","recovery_z_mean_diff2",

    "recovery_mean_roll4_mean","recovery_mean_roll4_std",

    "recovery_z_mean_roll4_mean","recovery_z_mean_roll4_std",



    # Social interaction (Social2)

    "social_mean","social_std","social_max","social_count",

    "social_z_mean","social_z_std","social_z_max",

    "social_mean_diff1","social_mean_diff2",

    "social_z_mean_diff1","social_z_mean_diff2",

    "social_mean_roll4_mean","social_mean_roll4_std",

    "social_z_mean_roll4_mean","social_z_mean_roll4_std",

]



# Add missingness flags (if present)

multimodal_cols += ["workload_missing","recovery_missing","social_missing"]



# Keep only columns that exist (not all datasets have all stats)

baseline_cols     = [c for c in baseline_cols     if c in multimodal_df.columns]

stress_sleep_cols = [c for c in stress_sleep_cols if c in multimodal_df.columns]

multimodal_cols   = [c for c in multimodal_cols   if c in multimodal_df.columns]



X_baseline     = multimodal_df[baseline_cols].copy()

X_stress_sleep = multimodal_df[stress_sleep_cols].copy()

X_multimodal   = multimodal_df[multimodal_cols].copy()



groups = multimodal_df["student_id"]

weeks  = multimodal_df["week"]



print("Feature sets ready:")

print("  A) Stress-only:     ", X_baseline.shape)

print("  B) Stress+Sleep:    ", X_stress_sleep.shape)

print("  C) Full Multimodal: ", X_multimodal.shape)

PREPARING FEATURE SETS
Feature sets ready:
  A) Stress-only:      (207, 7)
  B) Stress+Sleep:     (207, 12)
  C) Full Multimodal:  (207, 33)
